In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/bank.csv"
df = pd.read_csv(url)

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [2]:
df.shape


(45211, 17)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  deposit    45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 8.1 MB


In [4]:
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [5]:
df['deposit'].value_counts()

deposit
no     39922
yes     5289
Name: count, dtype: int64

In [8]:
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype == 'str':
        print(col)
        print(df[col].value_counts())
        print()

job
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
unknown           288
Name: count, dtype: int64

marital
marital
married     27214
single      12790
divorced     5207
Name: count, dtype: int64

education
education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64

default
default
no     44396
yes      815
Name: count, dtype: int64

housing
housing
yes    25130
no     20081
Name: count, dtype: int64

loan
loan
no     37967
yes     7244
Name: count, dtype: int64

contact
contact
cellular     29285
unknown      13020
telephone     2906
Name: count, dtype: int64

month
month
may    13766
jul     6895
aug     6247
jun     5341
nov     3970
apr     2932
feb     2649
jan     1403
oct      738
sep      579
mar      477
dec      214
Name: count,

In [9]:
for col in ['job', 'education', 'contact', 'poutcome']:
    unknown_count = (df[col] == 'unknown').sum()
    total = len(df)
    percent = round((unknown_count / total) * 100, 1)
    print(f"{col}: {unknown_count} unknowns out of {total} ({percent}%)")

job: 288 unknowns out of 45211 (0.6%)
education: 1857 unknowns out of 45211 (4.1%)
contact: 13020 unknowns out of 45211 (28.8%)
poutcome: 36959 unknowns out of 45211 (81.7%)


In [10]:
job_mode = df['job'].mode()[0]
df['job'] = df['job'].replace('unknown', job_mode)

education_mode = df['education'].mode()[0]
df['education'] = df['education'].replace('unknown', education_mode)

print(df['job'].value_counts())
print(df['education'].value_counts())

job
blue-collar      10020
management        9458
technician        7597
admin.            5171
services          4154
retired           2264
self-employed     1579
entrepreneur      1487
unemployed        1303
housemaid         1240
student            938
Name: count, dtype: int64
education
secondary    25059
tertiary     13301
primary       6851
Name: count, dtype: int64


In [11]:
df['was_contacted_before'] = df['pdays'].apply(lambda x: 0 if x == -1 else 1)

print(df['was_contacted_before'].value_counts())

was_contacted_before
0    36954
1     8257
Name: count, dtype: int64


In [12]:
print(df.duplicated().sum())

0


In [13]:
df['deposit'] = df['deposit'].map({'no': 0, 'yes': 1})

print(df['deposit'].value_counts())

deposit
0    39922
1     5289
Name: count, dtype: int64


In [14]:
binary_cols = ['default', 'housing', 'loan']

for col in binary_cols:
    df[col] = df[col].map({'no': 0, 'yes': 1})

print(df[binary_cols].head())

   default  housing  loan
0        0        1     0
1        0        1     0
2        0        1     1
3        0        1     0
4        0        0     0


In [15]:
nominal_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print(df.shape)
print(df.columns.tolist())

(45211, 42)
['age', 'default', 'balance', 'housing', 'loan', 'day', 'duration', 'campaign', 'pdays', 'previous', 'deposit', 'was_contacted_before', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'marital_married', 'marital_single', 'education_secondary', 'education_tertiary', 'contact_telephone', 'contact_unknown', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'poutcome_other', 'poutcome_success', 'poutcome_unknown']


In [16]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print(df[numeric_cols].head())

        age   balance       day  duration  campaign     pdays  previous
0  1.606965  0.256419 -1.298476  0.011016 -0.569351 -0.411453  -0.25194
1  0.288529 -0.437895 -1.298476 -0.416127 -0.569351 -0.411453  -0.25194
2 -0.747384 -0.446762 -1.298476 -0.707361 -0.569351 -0.411453  -0.25194
3  0.571051  0.047205 -1.298476 -0.645231 -0.569351 -0.411453  -0.25194
4 -0.747384 -0.447091 -1.298476 -0.233620 -0.569351 -0.411453  -0.25194


In [17]:
X = df.drop('deposit', axis=1)
y = df['deposit']

print(X.shape)
print(y.shape)

(45211, 41)
(45211,)


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

(36168, 41)
(9043, 41)


In [19]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000, class_weight='balanced')
log_model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [21]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)
tree_model.fit(X_train, y_train)

print("Decision tree trained successfully")

Decision tree trained successfully


In [22]:
log_predictions = log_model.predict(X_test)
tree_predictions = tree_model.predict(X_test)

print(log_predictions[:10])
print(tree_predictions[:10])

[0 0 0 0 0 0 0 1 0 0]
[0 0 0 0 0 1 0 1 0 0]


In [23]:
from sklearn.metrics import accuracy_score

log_accuracy = accuracy_score(y_test, log_predictions)
tree_accuracy = accuracy_score(y_test, tree_predictions)

print(f"Logistic Regression Accuracy: {log_accuracy:.4f}")
print(f"Decision Tree Accuracy: {tree_accuracy:.4f}")

Logistic Regression Accuracy: 0.8456
Decision Tree Accuracy: 0.7964


In [24]:
from sklearn.metrics import classification_report

print("Logistic Regression Report:")
print(classification_report(y_test, log_predictions))

print("Decision Tree Report:")
print(classification_report(y_test, tree_predictions))

Logistic Regression Report:
              precision    recall  f1-score   support

           0       0.97      0.85      0.91      7985
           1       0.42      0.81      0.55      1058

    accuracy                           0.85      9043
   macro avg       0.69      0.83      0.73      9043
weighted avg       0.91      0.85      0.87      9043

Decision Tree Report:
              precision    recall  f1-score   support

           0       0.98      0.79      0.87      7985
           1       0.35      0.85      0.50      1058

    accuracy                           0.80      9043
   macro avg       0.66      0.82      0.68      9043
weighted avg       0.90      0.80      0.83      9043



In [25]:
from sklearn.metrics import confusion_matrix

log_cm = confusion_matrix(y_test, log_predictions)
print("Logistic Regression Confusion Matrix:")
print(log_cm)

Logistic Regression Confusion Matrix:
[[6785 1200]
 [ 196  862]]


In [26]:
results = X_test.copy()
results['actual_deposit'] = y_test.values
results['predicted_deposit_logistic'] = log_predictions
results['predicted_deposit_tree'] = tree_predictions

results.to_csv('predictions.csv', index=False)

print("Predictions saved to predictions.csv")
print(results.head())

Predictions saved to predictions.csv
            age  default   balance  housing  loan       day  duration  \
1392  -0.088167        0 -0.237220        1     1 -0.938003  0.344964   
7518   0.288529        0 -0.323270        1     0  1.705471 -0.214205   
12007 -0.935732        0 -0.330496        1     0  0.503892 -0.117127   
5536  -0.464863        0 -0.232294        1     0  0.864365 -0.408361   
29816 -0.653211        0  0.183506        1     0 -1.418634 -0.765608   

       campaign     pdays  previous  ...  month_may  month_nov  month_oct  \
1392  -0.246560 -0.411453  -0.25194  ...       True      False      False   
7518  -0.246560 -0.411453  -0.25194  ...       True      False      False   
12007  0.721811 -0.411453  -0.25194  ...      False      False      False   
5536   0.399020 -0.411453  -0.25194  ...       True      False      False   
29816 -0.569351 -0.411453  -0.25194  ...      False      False      False   

       month_sep  poutcome_other  poutcome_success  poutcome_

In [26]:
import joblib
import json

joblib.dump(log_model, 'log_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

model_columns = X.columns.tolist()
with open('model_columns.json', 'w') as f:
    json.dump(model_columns, f)

print("Saved: log_model.pkl, scaler.pkl, model_columns.json")

Saved: log_model.pkl, scaler.pkl, model_columns.json
